# Auto S-tune  *(thin notebook over the `AutoSQUID` package)*

**Live-center** the locked input-SQUID output near a target (default 0 mV) by stepping the SQUID-flux DAC —
automating the manual "nudge S-flux until the trace looks centered" procedure (`sq.auto_s_tune`, a secant
search on short live-mean reads). Run on the **bench PC**. No reset between steps (flux shifts the locked
baseline directly).

**Lock the input SQUID first** (`sq.s_lock(cfg)` below, or in PCS102DA) and make sure the trace is roughly
centered (within ±1 V), then run §1.

## §0 — Config  *(the cell you edit)*

In [ ]:
# add the package's PARENT to sys.path so `import AutoSQUID` works from inside the AutoSQUID/ folder,
# and the bench-PC RanLabPythonRepo (two levels up, like the other notebooks) is importable.
import sys
sys.path.insert(0, "..")          # .../automation  -> exposes the `AutoSQUID` package
sys.path.append("../../")         # .../SQUID/... root for RanLabPythonRepo (same as the other notebooks)
import AutoSQUID as sq
print("AutoSQUID loaded:", [n for n in sq.__all__[:6]], "...")

cfg = sq.Config(
    port    = "COM3",     # SCC control port
    channel = 1,          # locked PFL channel
    daq_ai  = "Dev1/ai0", # NI analog-in carrying the locked output
    vrange  = 1.0,        # +/-1 V AI range (start must be on-scale)
)
print(f"control PORT={cfg.port} ch={cfg.channel} · DAQ_AI={cfg.daq_ai} · vrange=+/-{cfg.vrange} V")

## §1 — Lock + auto S-tune  *(ACTIVE — sends to the hardware)*

Close PCS102DA on this port first. `auto_s_tune` returns a dict: `status` ∈
{converged, no_response, max_iter}, plus `flux_sflux` / `mean_V` / `std_V` / `n_iter`.
Tunables: `tol_V` (centered band, default 3 mV), `read_s` (live-view length), `start_sflux`, `max_step_uA`.

In [ ]:
sq.s_lock(cfg)                                   # lock the input SQUID (or do it in PCS102DA)
res = sq.auto_s_tune(cfg, start_sflux=50.0, target_V=0.0, tol_V=0.003)
print(res)
if res["status"] != "converged":
    print("NOT centered — nudge S-flux manually closer / check lock, then re-run.")